# Phase 4 — Full MIL Grid

**Two parts:**

**Part A: MIL ablation** — `All` streams, `all10` policy, 5 LOYO folds × 4 conditions:
- `MIL_resolved`  — phase-resolved attention (main model)
- `MIL_shared`    — phase-shared attention (one weight per field, shared across phases)
- `mean_pool`     — uniform 1/10 weights (no attention)
- `rank1`         — train on closest field only, no MIL

**Part B: Stream ablation** — for the winning condition from Part A, run all 5 stream configs.

Total: 5×4 + 5×5 = **45 model trainings**. Resumable.

Outputs to `phase4_outputs/`.


## 1. Setup

In [ ]:
from pathlib import Path
import sys, os, time, gc, psutil
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
OUTPUT_DIR  = PROJECT_ROOT / 'phase4_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === KNOBS ===
YEARS       = [2017, 2018, 2019, 2020, 2021]
EPOCHS      = 15
HIDDEN      = 32
BATCH_SIZE  = 8     # bags per minibatch (each bag = up to 10 fields)
LR          = 1e-3
USE_AMP     = True


## 2. Helpers (fold loading, evaluation, resumable persistence)

In [ ]:
from phase3_baselines.dataset import (load_year_bundle, SelectionPolicy,
                                       apply_selection, concat_bundles)
from phase3_baselines.evaluate import evaluate_predictions
from phase3_baselines.resumable import append_row, already_done, save_pred_npz
from phase4_mil.bag_dataset import build_bags
from phase4_mil.train import train_mil_fold, STREAM_CONFIGS


def load_loyo_bags(test_year, use_rank1=False):
    """Load 4 train years + 1 test year, build bags."""
    policy = SelectionPolicy(use_only_rank1=use_rank1)
    test_b = apply_selection(
        load_year_bundle(TENSORS_DIR / f'phenology_{test_year}.npz'), policy)
    train_parts = {}
    for y in YEARS:
        if y == test_year: continue
        train_parts[y] = apply_selection(
            load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
    train_b = concat_bundles(train_parts, years=[y for y in YEARS if y != test_year])
    del train_parts; gc.collect()
    return build_bags(train_b), build_bags(test_b)


def record_metrics(csv_path, condition, stream, test_year, result, phase_ids):
    """Append both station- and field-level per-phase rows to csv."""
    rep_st = evaluate_predictions(
        pred_days=result['station_pred'].astype(np.float32),
        true_days=result['station_truth'].astype(np.float32),
        phase_ids=phase_ids, years=result['station_year'], ranks=None,
    )
    rep_fd = evaluate_predictions(
        pred_days=result['field_pred'].astype(np.float32),
        true_days=result['field_truth'].astype(np.float32),
        phase_ids=phase_ids, years=result['field_year'],
        ranks=result['field_rank'],
    )
    for phase, m in rep_st['per_phase'].iterrows():
        append_row(csv_path, {
            'condition': condition, 'stream': stream,
            'test_year': int(test_year), 'level': 'station',
            'phase': int(phase), **m.to_dict()
        })
    for phase, m in rep_fd['per_phase'].iterrows():
        append_row(csv_path, {
            'condition': condition, 'stream': stream,
            'test_year': int(test_year), 'level': 'field',
            'phase': int(phase), **m.to_dict()
        })


## 3. Part A: MIL ablation (All streams, all10 policy)

Run 4 conditions × 5 LOYO folds = 20 fits.


In [ ]:
RESULTS_CSV = OUTPUT_DIR / 'mil_partA_ablation.csv'
PREDS_NPZ   = OUTPUT_DIR / 'mil_partA_predictions.npz'

# Condition definitions
PART_A_CONDITIONS = [
    # (condition_name, phase_resolved_attention, use_rank1)
    ('MIL_resolved', True,  False),
    ('MIL_shared',   False, False),
    ('mean_pool',    None,  False),   # special-case below
    ('rank1',        False, True),    # only 1 field per bag → MIL inactive
]

done = already_done(RESULTS_CSV, ['condition', 'stream', 'test_year'])
print(f'Done: {len(done)} of {len(PART_A_CONDITIONS) * len(YEARS)} fits')

# Bag cache to avoid reloading
bag_cache = {}
def get_bags(test_year, use_rank1):
    key = (test_year, use_rank1)
    if key not in bag_cache:
        bag_cache[key] = load_loyo_bags(test_year, use_rank1=use_rank1)
    return bag_cache[key]

pbar = tqdm(total=len(PART_A_CONDITIONS) * len(YEARS), desc='Part A')

for cond_name, pr_attn, use_r1 in PART_A_CONDITIONS:
    for test_y in YEARS:
        key = (cond_name, 'All', int(test_y))
        if key in done:
            pbar.update(1); continue
        t0 = time.time()
        train_bags, test_bags = get_bags(test_y, use_rank1=use_r1)

        if cond_name == 'mean_pool':
            # Force uniform attention by monkey-patching post-init.
            # Implementation: we use phase_resolved=False then zero out the scorer's
            # final layer so all scores are identical → softmax = uniform.
            import torch.nn as nn
            # Special path: train with phase_resolved=False, then override scorer
            # to make all scores constant. Easiest: use a model variant.
            from phase4_mil.model import MILModel
            model_kwargs = dict(phase_resolved_attention=False)
            class MeanPoolModel(MILModel):
                def forward(self, batch, active=('S1','S2','Met')):
                    # Override: compute uniform alpha then run rest of the network.
                    X = batch['X']; fmask = batch['feat_mask']; bmask = batch['bag_mask']
                    h_seq = self.encode_bag(X, fmask, active=active)
                    # h_field unused for alpha, but field head still uses h_seq
                    # Build uniform alpha
                    bm = bmask.float()                            # (b, B)
                    counts = bm.sum(dim=1, keepdim=True).clamp(min=1.0)
                    alpha_unif = (bm / counts).unsqueeze(-1).expand(-1, -1, self.n_phases)
                    z = torch.einsum('bjth,bjp->btph', h_seq, alpha_unif)
                    station_logits = torch.einsum('btph,ph->btp', z, self.station_head) + self.station_bias
                    field_logits = self.field_head(h_seq)
                    return {'station_logits': station_logits,
                            'field_logits':   field_logits,
                            'alpha':          alpha_unif,
                            'h_field':        h_seq.mean(dim=2)}
            # Hack: monkey-patch the symbol used inside train_mil_fold by replacing
            # the global MILModel reference temporarily.
            from phase4_mil import train as _train_mod
            from phase4_mil import model as _model_mod
            _orig = _model_mod.MILModel
            _model_mod.MILModel = MeanPoolModel
            _train_mod.MILModel = MeanPoolModel
            try:
                result = train_mil_fold(
                    train_bags=train_bags, test_bags=test_bags,
                    active=STREAM_CONFIGS['All'],
                    hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                    phase_resolved_attention=False, field_loss_weight=0.3,
                    use_amp=USE_AMP, device=DEVICE, verbose=False,
                )
            finally:
                _model_mod.MILModel = _orig
                _train_mod.MILModel = _orig
        else:
            result = train_mil_fold(
                train_bags=train_bags, test_bags=test_bags,
                active=STREAM_CONFIGS['All'],
                hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                phase_resolved_attention=(pr_attn if pr_attn is not None else False),
                field_loss_weight=0.3,
                use_amp=USE_AMP, device=DEVICE, verbose=False,
            )

        dt = time.time() - t0
        phase_ids = train_bags['phase_ids']
        record_metrics(RESULTS_CSV, cond_name, 'All', test_y, result, phase_ids)

        # Save predictions + attention
        save_pred_npz(PREDS_NPZ,
                      f'{cond_name}__All__{test_y}',
                      st_pred=result['station_pred'].astype(np.int32),
                      st_truth=result['station_truth'].astype(np.int32),
                      st_bag=result['station_bag_id'],
                      fd_pred=result['field_pred'].astype(np.int32),
                      fd_truth=result['field_truth'].astype(np.int32),
                      fd_bag=result['field_bag_id'],
                      fd_rank=result['field_rank'],
                      alpha=result['alpha'].astype(np.float32),
                      alpha_bag=result['alpha_bag_id'])

        pbar.set_postfix({'last': f'{cond_name}/{test_y}', 'secs': f'{dt:.0f}'})
        pbar.update(1)

pbar.close()
print('Part A complete.')


## 4. Part A summary

In [ ]:
results = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(results)} rows')

# Headline: macro MAE per (condition, level)
summary = (results.groupby(['condition','level'])
                   .agg(macro_mae=('mae','mean'),
                        macro_r2 =('r2','mean'),
                        macro_within_5d=('within_5d','mean'))
                   .reset_index()
                   .sort_values(['level','macro_mae']))
print('\n=== Part A: Condition comparison ===')
print(summary.round(3).to_string(index=False))


In [ ]:
# Per-phase MAE for station-level by condition
station = results[results['level']=='station']
table = station.pivot_table(index='phase', columns='condition', values='mae', aggfunc='mean').round(2)
print('\n=== Station-level MAE by (condition, phase) ===')
print(table.to_string())

# Field-level
field = results[results['level']=='field']
table_f = field.pivot_table(index='phase', columns='condition', values='mae', aggfunc='mean').round(2)
print('\n=== Field-level MAE by (condition, phase) ===')
print(table_f.to_string())


## 5. Part B: Stream ablation (winning condition from Part A)

Look at Part A output. The winning condition is the one with lowest station-level macro_mae.
Set `WINNING_CONDITION` below accordingly, then run.

Default below assumes `MIL_resolved` won. Change if needed.


In [ ]:
WINNING_CONDITION = 'MIL_resolved'    # change based on Part A results

PART_B_CSV  = OUTPUT_DIR / 'mil_partB_streams.csv'
PART_B_NPZ  = OUTPUT_DIR / 'mil_partB_predictions.npz'

PART_B_STREAMS = ['S1_only', 'S2_only', 'Met_only', 'S1+S2', 'All']

# Map winning condition to model kwargs
def kwargs_for(cond):
    if cond == 'MIL_resolved': return dict(phase_resolved_attention=True, use_rank1=False)
    if cond == 'MIL_shared':   return dict(phase_resolved_attention=False, use_rank1=False)
    if cond == 'mean_pool':    return dict(phase_resolved_attention=None, use_rank1=False)  # not handled here
    if cond == 'rank1':        return dict(phase_resolved_attention=False, use_rank1=True)
    raise ValueError(cond)

cfg = kwargs_for(WINNING_CONDITION)
if cfg['phase_resolved_attention'] is None:
    raise RuntimeError('mean_pool needs the monkey-patch path; only run if Part A said it won.')

done_B = already_done(PART_B_CSV, ['condition', 'stream', 'test_year'])
print(f'Done: {len(done_B)} of {len(PART_B_STREAMS) * len(YEARS)} fits')

pbar = tqdm(total=len(PART_B_STREAMS) * len(YEARS), desc='Part B')
for stream in PART_B_STREAMS:
    for test_y in YEARS:
        key = (WINNING_CONDITION, stream, int(test_y))
        if key in done_B:
            pbar.update(1); continue
        t0 = time.time()
        train_bags, test_bags = get_bags(test_y, use_rank1=cfg['use_rank1'])
        result = train_mil_fold(
            train_bags=train_bags, test_bags=test_bags,
            active=STREAM_CONFIGS[stream],
            hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
            phase_resolved_attention=cfg['phase_resolved_attention'],
            field_loss_weight=0.3,
            use_amp=USE_AMP, device=DEVICE, verbose=False,
        )
        dt = time.time() - t0
        phase_ids = train_bags['phase_ids']
        record_metrics(PART_B_CSV, WINNING_CONDITION, stream, test_y, result, phase_ids)
        save_pred_npz(PART_B_NPZ,
                      f'{WINNING_CONDITION}__{stream}__{test_y}',
                      st_pred=result['station_pred'].astype(np.int32),
                      st_truth=result['station_truth'].astype(np.int32),
                      st_bag=result['station_bag_id'],
                      fd_pred=result['field_pred'].astype(np.int32),
                      fd_truth=result['field_truth'].astype(np.int32),
                      fd_bag=result['field_bag_id'],
                      fd_rank=result['field_rank'],
                      alpha=result['alpha'].astype(np.float32),
                      alpha_bag=result['alpha_bag_id'])
        pbar.set_postfix({'last': f'{stream}/{test_y}', 'secs': f'{dt:.0f}'})
        pbar.update(1)
pbar.close()
print('Part B complete.')


## 6. Part B summary

In [ ]:
results_B = pd.read_csv(PART_B_CSV)
print(f'Loaded {len(results_B)} rows')

# Headline by stream, level
summary_B = (results_B.groupby(['stream','level'])
                      .agg(macro_mae=('mae','mean'),
                           macro_r2 =('r2','mean'),
                           macro_within_5d=('within_5d','mean'))
                      .reset_index()
                      .sort_values(['level','macro_mae']))
print('\n=== Part B: Stream comparison ===')
print(summary_B.round(3).to_string(index=False))


In [ ]:
# Per-phase for station-level
table_b = (results_B[results_B['level']=='station']
                       .pivot_table(index='phase', columns='stream', values='mae', aggfunc='mean').round(2))
print('\n=== Part B station-level MAE by (stream, phase) ===')
print(table_b.to_string())


## 7. Compare to Phase 3 LSTM

In [ ]:
# Load Phase 3 LSTM headline for direct comparison
P3 = pd.read_csv(PROJECT_ROOT / 'phase3_outputs/lstm/lstm_results.csv')
p3_all = (P3[(P3['stream']=='All') & (P3['policy']=='all10')]
            .groupby('phase')['mae'].mean().round(2))
print('\n=== Phase 3 LSTM (All / all10), station-level proxy ===')
print(p3_all.to_string())

p4_all = (results_B[(results_B['stream']=='All') & (results_B['level']=='station')]
            .groupby('phase')['mae'].mean().round(2))
print('\n=== Phase 4 MIL ('+ WINNING_CONDITION + ' / All), station-level ===')
print(p4_all.to_string())

print('\n=== Difference (P4 − P3); negative = MIL improved ===')
print((p4_all - p3_all).to_string())


---\n**Done with Phase 4.**